# BundleRecommender — UCI Product Names

## Standalone extraction (no retraining)

**Problem fixed:** `/kaggle/working/bundlerecommender_master_v1/models/uci_rules.csv` may not exist in a fresh Kaggle session. This notebook embeds only the 46 verified SKU identifiers from the project's existing rule artifact. Product *descriptions* are read exclusively from the original Online Retail II workbook.

**Inputs:** Kaggle source workbook at `/kaggle/input/datasets/wafabila/bundlerecommender-dataset/onlineretailii/online_retail_II.xlsx`.

**Outputs:** `/kaggle/working/uci_product_catalog.csv`, `/kaggle/working/uci_product_catalog_audit.json`.

**Boundary:** No training, no recalculation of rules, metrics, or labels. Unknown SKUs retain the UI fallback behavior.

## 01 — Source check

The source must exist. The catalog is filtered to SKU identities present in `models/uci_rules.csv` on GitHub as inspected on 26 October 2026. If the rule artifact changes, regenerate the allowlist from the updated CSV.

In [1]:
from __future__ import annotations

import csv
import json
from collections import Counter, defaultdict
from pathlib import Path
from datetime import datetime, timezone

try:
    from openpyxl import load_workbook
except ImportError as exc:
    raise RuntimeError("Install openpyxl in Kaggle: %pip install openpyxl") from exc

SOURCE = Path("/kaggle/input/datasets/wafabila/bundlerecommender-dataset/onlineretailii/online_retail_II.xlsx")
OUTPUT_DIR = Path("/kaggle/working")
OUTPUT_CSV = OUTPUT_DIR / "uci_product_catalog.csv"
OUTPUT_AUDIT = OUTPUT_DIR / "uci_product_catalog_audit.json"

# 46 StockCodes extracted directly from:
# https://github.com/MENSTRUE/BundleRecommender/blob/main/models/uci_rules.csv
# This list is an IDENTIFIER ALLOWLIST, not generated product descriptions.
RULE_SKUS = set("20712 20713 20719 20723 20724 20725 20726 20727 20728 21080 21212 21213 21231 21232 21733 21754 21755 21928 21929 21931 21975 21977 22355 22356 22379 22382 22383 22384 22386 22411 22423 22469 22470 22629 22630 22697 22699 82482 82494L 84879 84991 84992 85099B 85099C 85099F 85123A".split())

def normalize_header(value: object) -> str:
    return "".join(ch for ch in str(value or "").lower() if ch.isalnum())

def clean_value(value: object) -> str:
    return "" if value is None else str(value).strip()

def run() -> dict:
    if not SOURCE.is_file():
        raise FileNotFoundError(f"Original UCI workbook not found: {SOURCE}")
    if len(RULE_SKUS) != 46:
        raise RuntimeError("Unexpected SKU allowlist length; stop rather than mislabel data.")

    mapping: dict[str, Counter[str]] = defaultdict(Counter)
    scanned = 0
    matched = 0
    sheet_stats = []
    book = load_workbook(SOURCE, read_only=True, data_only=True)

    try:
        for sheet in book.worksheets:
            rows = sheet.iter_rows(values_only=True)
            header = next(rows, None)
            names = [normalize_header(v) for v in header] if header else []
            if "stockcode" not in names or "description" not in names:
                print(f"SKIP sheet {sheet.title!r}: no StockCode/Description columns")
                continue

            sku_index = names.index("stockcode")
            desc_index = names.index("description")
            sheet_seen = 0
            sheet_matches = 0
            for values in rows:
                sheet_seen += 1
                scanned += 1
                if len(values) <= max(sku_index, desc_index):
                    continue
                sku = clean_value(values[sku_index]).upper()
                if sku not in RULE_SKUS:
                    continue
                desc = " ".join(clean_value(values[desc_index]).split())
                if not desc or desc.lower() in {"nan", "none", "null"}:
                    continue
                mapping[sku][desc] += 1
                sheet_matches += 1
                matched += 1
            sheet_stats.append({"sheet": sheet.title, "rows": sheet_seen, "matching_named_rows": sheet_matches})
            print(f"{sheet.title}: scanned {sheet_seen:,} rows, matched {sheet_matches:,} named observations")
    finally:
        book.close()

    if not sheet_stats:
        raise RuntimeError("No worksheet has StockCode and Description columns.")
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    missing = []
    with OUTPUT_CSV.open("w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["stock_code", "description"])
        writer.writeheader()
        for sku in sorted(RULE_SKUS):
            candidates = mapping.get(sku)
            if not candidates:
                missing.append(sku)
                continue
            chosen = sorted(candidates, key=lambda value: (-candidates[value], value))[0]
            writer.writerow({"stock_code": sku, "description": chosen})

    audit = {
        "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        "dataset_source": str(SOURCE),
        "sku_allowlist_source": "MENSTRUE/BundleRecommender/models/uci_rules.csv (GitHub main, inspected 2026-10-26)",
        "rule_skus": len(RULE_SKUS),
        "matched_named_skus": len(RULE_SKUS) - len(missing),
        "unmatched_skus": missing,
        "rows_scanned": scanned,
        "matching_named_observations": matched,
        "sheets": sheet_stats,
        "selection_rule": "Most frequently observed nonblank Description per StockCode, lexicographic tie-break",
        "caution": "Names are metadata only; no model training or association-rule changes performed."
    }
    OUTPUT_AUDIT.write_text(json.dumps(audit, indent=2, ensure_ascii=False), encoding="utf-8")
    print("\nSAVED:", OUTPUT_CSV)
    print("NAMED:", audit["matched_named_skus"], "/", len(RULE_SKUS))
    print("UNMATCHED:", missing)
    print("PREVIEW:", OUTPUT_CSV.read_text(encoding="utf-8").splitlines()[:7])
    return audit

audit = run()


Year 2009-2010: scanned 525,461 rows, matched 52,202 named observations
Year 2010-2011: scanned 541,910 rows, matched 49,202 named observations

SAVED: /kaggle/working/uci_product_catalog.csv
NAMED: 46 / 46
UNMATCHED: []
PREVIEW: ['stock_code,description', '20712,JUMBO BAG WOODLAND ANIMALS', '20713,JUMBO BAG OWLS', '20719,WOODLAND CHARLOTTE BAG', '20723,STRAWBERRY CHARLOTTE BAG', '20724,RED RETROSPOT CHARLOTTE BAG', '20725,LUNCH BAG RED RETROSPOT']


## 02 — Verify and download

Confirm `matched_named_skus` and inspect the generated CSV. Add `uci_product_catalog.csv` to local `models/` then restart FastAPI or refresh the page as needed. No other artifact is modified.

In [2]:
from IPython.display import display
import pandas as pd
display(pd.read_csv("/kaggle/working/uci_product_catalog.csv").head(15))
print("Total mapped names:", audit["matched_named_skus"])
print("Unmatched SKU count:", len(audit["unmatched_skus"]))

,stock_code,description
0,20712,JUMBO BAG WOODLAND ANIMALS
1,20713,JUMBO BAG OWLS
2,20719,WOODLAND CHARLOTTE BAG
3,20723,STRAWBERRY CHARLOTTE BAG
4,20724,RED RETROSPOT CHARLOTTE BAG
5,20725,LUNCH BAG RED RETROSPOT
6,20726,LUNCH BAG WOODLAND
7,20727,LUNCH BAG BLACK SKULL.
8,20728,LUNCH BAG CARS BLUE
9,21080,SET/20 RED RETROSPOT PAPER NAPKINS


Total mapped names: 46
Unmatched SKU count: 0
